In [ ]:
import os
import gspread
import pandas as pd
import yfinance as yf
import numpy as np
import requests
from datetime import datetime, timedelta

# ==========================================
# CONFIGURAÇÕES DO USUÁRIO (PREENCHA AQUI)
# ==========================================
TELEGRAM_BOT_TOKEN = 'SEU_TOKEN_DO_TELEGRAM_AQUI'
TELEGRAM_CHAT_ID = 'SEU_CHAT_ID_AQUI'
GOOGLE_SHEETS_ID = 'ID_DA_SUA_PLANILHA_AQUI'
DIRETORIO_CREDENCIAIS = r"CAMINHO_DA_SUA_PASTA_LOCAL_AQUI"

# ==========================================
# 1. FUNÇÕES BASE (RSI e TELEGRAM)
# ==========================================
def calcular_rsi(dados_preco, periodos=14):
    delta = dados_preco.diff()
    ganho = (delta.where(delta > 0, 0)).rolling(window=periodos).mean()
    perda = (-delta.where(delta < 0, 0)).rolling(window=periodos).mean()
    rs = ganho / perda
    return 100 - (100 / (1 + rs))

def enviar_mensagem_telegram(mensagem):
    """Envia a mensagem formatada para o Telegram"""
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    payload = {"chat_id": TELEGRAM_CHAT_ID, "text": mensagem, "parse_mode": "HTML"}

    try:
        response = requests.post(url, json=payload)
        if response.status_code == 200:
            print("📱 Mensagem enviada com sucesso para o Telegram!")
        else:
            print(f"⚠️ Erro ao enviar para o Telegram: {response.text}")
    except Exception as e:
        print(f"⚠️ Falha na conexão com o Telegram: {e}")

# ==========================================
# 2. MOTOR DE ANÁLISE DO RADAR
# ==========================================
def rodar_radar_oportunidades(carteira_atual):
    print(f"\n[{datetime.now().strftime('%d/%m/%Y %H:%M')}] Iniciando Radar de Oportunidades...\n")

    alertas = []
    data_inicio = (datetime.now() - timedelta(days=60)).strftime('%Y-%m-%d')

    if not carteira_atual:
        print("⚠️ AVISO: A lista de ações está vazia.")
        return

    for ativo in carteira_atual:
        ticker = ativo['ticker']
        preco_medio = ativo['preco_medio']
        ticker_yf = ticker if str(ticker).upper().endswith('.SA') else f"{ticker}.SA"

        try:
            df = yf.download(ticker_yf, start=data_inicio, progress=False)
            if df.empty:
                continue

            preco_atual = float(df['Close'].iloc[-1].iloc[0] if isinstance(df['Close'].iloc[-1], pd.Series) else df['Close'].iloc[-1])
            df['RSI'] = calcular_rsi(df['Close'])
            rsi_atual = float(df['RSI'].iloc[-1].iloc[0] if isinstance(df['RSI'].iloc[-1], pd.Series) else df['RSI'].iloc[-1])
            distancia_pm = ((preco_atual / preco_medio) - 1) * 100

            if preco_atual < preco_medio and rsi_atual <= 35:
                alertas.append({'Ticker': ticker, 'Sinal': '🟢 COMPRA', 'Preco': preco_atual, 'PM': preco_medio, 'Distancia': distancia_pm, 'RSI': rsi_atual})
            elif preco_atual > (preco_medio * 1.10) and rsi_atual >= 70:
                 alertas.append({'Ticker': ticker, 'Sinal': '🔴 VENDA', 'Preco': preco_atual, 'PM': preco_medio, 'Distancia': distancia_pm, 'RSI': rsi_atual})

        except Exception as e:
            print(f"⚠️ Erro ao processar {ticker}: {e}")

    # ==========================================
    # 3. EXIBIÇÃO E ENVIO DO ALERTA
    # ==========================================
    if not alertas:
        print("⚖️ Nenhuma oportunidade tática detectada hoje.")
    else:
        print(f"🚨 {len(alertas)} OPORTUNIDADE(S) DETECTADA(S) 🚨")
        mensagem_telegram = "🚨 <b>RADAR B3 - OPORTUNIDADES</b> 🚨\n\n"

        for alerta in alertas:
            mensagem_telegram += f"<b>{alerta['Ticker']}</b> ({alerta['Sinal']})\n"
            mensagem_telegram += f"Atual: R$ {alerta['Preco']:.2f} | Seu PM: R$ {alerta['PM']:.2f}\n"
            mensagem_telegram += f"Distância: {alerta['Distancia']:.2f}% | RSI: {alerta['RSI']:.1f}\n\n"

        mensagem_telegram += "<i>Análise gerada automaticamente.</i>"
        enviar_mensagem_telegram(mensagem_telegram)

# ==========================================
# 4. CONEXÃO COM O GOOGLE SHEETS E TRANSFORMAÇÃO
# ==========================================
def buscar_carteira_do_sheets():
    try:
        import google.colab
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False

    if IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive')
        diretorio_base = '/content/drive/MyDrive/' # Ajuste para o seu caminho no Colab
    else:
        diretorio_base = DIRETORIO_CREDENCIAIS

    if not os.path.exists(diretorio_base):
        raise FileNotFoundError(f"🚨 ERRO: A pasta {diretorio_base} não foi encontrada.")

    arquivos_json = [f for f in os.listdir(diretorio_base) if f.endswith('.json')]

    if not arquivos_json:
        raise FileNotFoundError(f"🚨 ERRO: Nenhum arquivo .json encontrado na pasta {diretorio_base}.")

    nome_do_arquivo = arquivos_json[0]
    caminho_credenciais = os.path.join(diretorio_base, nome_do_arquivo)

    gc = gspread.service_account(filename=caminho_credenciais)
    sh = gc.open_by_key(GOOGLE_SHEETS_ID)
    worksheet = sh.worksheet('ControleAcoes')

    dados = worksheet.get_all_records()
    df_transacoes = pd.DataFrame(dados)

    df_transacoes['Tipo'] = df_transacoes['Tipo'].astype(str).str.strip().str.upper()
    df_compras = df_transacoes[df_transacoes['Tipo'] == 'COMPRA'].copy()

    df_compras['Qtd'] = pd.to_numeric(df_compras['Qtd'], errors='coerce')
    df_compras['Preco_Unitario'] = pd.to_numeric(df_compras['Preco_Unitario'], errors='coerce')

    df_compras['Volume'] = df_compras['Qtd'] * df_compras['Preco_Unitario']
    agrupado = df_compras.groupby('Ativo').agg({'Qtd': 'sum', 'Volume': 'sum'}).reset_index()
    agrupado['preco_medio'] = agrupado['Volume'] / agrupado['Qtd']

    carteira_real = []
    for index, row in agrupado.iterrows():
        carteira_real.append({'ticker': row['Ativo'], 'preco_medio': row['preco_medio']})

    return carteira_real

if __name__ == "__main__":
    try:
        minha_carteira_real = buscar_carteira_do_sheets()
        rodar_radar_oportunidades(minha_carteira_real)
    except Exception as e:
        print(f"\nErro fatal na execução: {e}")